# Replan - Adaptive Semester Planner

Replan is an App which will Plan your self study hours from today to the exam date according to your course's required credit hours and outline.

*Missed a Scheduled Task? No worries! It will reschedule.*

### Step 1: ask the exam date

In [53]:
user_input = input("Exam date (YYYY-MM-DD): ")

convert string input into date object

In [54]:
from datetime import datetime
import pandas as pd

# Parse the user's exam date flexibly (accepts YYYY-MM-DD, commas, slashes, etc.)
try:
    exdate = pd.to_datetime(user_input.strip()).to_pydatetime()
    print("Parsed Exam Date:", exdate.strftime("%Y-%m-%d %H:%M:%S"))
except Exception:
    try:
        exdate = datetime.strptime(user_input.strip(), '%Y-%m-%d')
        print("Parsed Exam Date:", exdate.strftime("%Y-%m-%d %H:%M:%S"))
    except ValueError:
        print(f"Error: Unable to parse '{user_input}'. Please use the format YYYY-MM-DD (e.g., 2027-02-02).")
        raise

Parsed Exam Date: 2026-10-17 00:00:00


### Step 2: Count days left from today till exam date

In [55]:
from datetime import datetime

# Compare calendar dates directly to avoid time truncation and type mismatch errors
today = datetime.today().date()
exam_date = exdate.date() if isinstance(exdate, datetime) else exdate

daysleft = (exam_date - today).days
print(f"Days left until exam: {daysleft}")

Days left until exam: 1


we got remaining days from today till the exam date: daysleft

# Step 3: Taking input credit hours for a course and divide it into the daysleft

In [56]:
CH_input = input("Total Credit Hours: ")

try:
    CH = float(CH_input)
    print(f"Total Credit Hours: {CH}")
except ValueError:
    print(f"Error: '{CH_input}' is not a valid number. Please provide a numeric value (e.g., 3).")
    CH = 0.0

Total Credit Hours: 3.0


Converting CHs into Clock(study) hours <br>
1 CH = 45 study hours

In [57]:
CH = float(CH)
totalStudyHours = CH * 45
print(totalStudyHours)

135.0


dividing these hours into the daysleft for the exam

In [58]:
CH = float(CH)
totalStudyHours = CH * 45

if 'daysleft' in locals() and daysleft > 0:
    hourspday = totalStudyHours / daysleft
    print("Days left:", daysleft)
    print("Hours needed to study:", totalStudyHours)
    print(f"Daily Study hours: {hourspday:.2f}")
else:
    hourspday = 0.0
    print("⚠️ No study days remaining before the exam date!")

Days left: 1
Hours needed to study: 135.0
Daily Study hours: 135.00


### Step 4: Course Outline & Topic Allocation

In [59]:
outline_input = input("Enter course topics (separated by commas): ")
# Example input: Introduction & Syntax, Data Structures, OOP, File Handling, Project & Revision

outline_input = outline_input.split(',')
topics = list()
for t in outline_input:
    topic = t.strip()
    if topic: topics.append(topic)

print(topics)  

if topics:
    hoursptopic = totalStudyHours / len(topics)
    hoursptopic = round(hoursptopic)
    print("Overall Study time per topic =",hoursptopic)

    print(" ")
    print("Overall Study Time per Topic:")
    i = 1
    for topic in topics:
        print(i, ".", topic, "=", hoursptopic, "hrs")
        i +=1
else:
    print("Enter Topics Please")

['Introduction & Syntax', 'Data Structures', 'OOP', 'File Handling', 'Project & Revision']
Overall Study time per topic = 27
 
Overall Study Time per Topic:
1 . Introduction & Syntax = 27 hrs
2 . Data Structures = 27 hrs
3 . OOP = 27 hrs
4 . File Handling = 27 hrs
5 . Project & Revision = 27 hrs


### Step 5: Generate Day-by-Day Schedule
(Divide the hoursptopic into the days between today and the exam date)

In [60]:
from datetime import datetime, timedelta
import pandas as pd

schedule_rows = []
curr_date = datetime.today().date()
exam_date = exdate.date() if isinstance(exdate, datetime) else exdate

remDays = (exam_date - curr_date).days
if 'hourspday' in locals() and hourspday > 0:
    daily_hours = round(hourspday, 2)
elif remDays > 0 and 'totalStudyHours' in locals():
    daily_hours = round((totalStudyHours / remDays), 2)
else:
    daily_hours = 0.0

if len(topics) > 0 and remDays > 0:
    num_topics = len(topics)
    base_days = remDays // num_topics
    extra_days = remDays % num_topics
    
    # Distribute the available days evenly across topics
    topic_durations = []
    for i in range(num_topics):
        if i < extra_days:
            topic_durations.append(base_days + 1)
        else:
            topic_durations.append(base_days)

    topic_periods = []
    day_counter = 0
    
    # Generate schedule and topic period distribution
    for idx, topic in enumerate(topics):
        duration = topic_durations[idx]
        start_date = curr_date + timedelta(days=day_counter)
        end_date = curr_date + timedelta(days=day_counter + max(0, duration - 1))
        topic_hours = round(duration * daily_hours, 1)
        
        topic_periods.append({
            "Topic #": idx + 1,
            "Topic": topic,
            "Duration": f"{duration} days",
            "Total Hours": f"{topic_hours} hrs",
            "Start Date": start_date.strftime("%Y-%m-%d"),
            "End Date": end_date.strftime("%Y-%m-%d")
        })
        
        for d in range(duration):
            day_date = curr_date + timedelta(days=day_counter)
            schedule_rows.append({
                "Day": day_counter + 1,
                "Date": day_date.strftime("%Y-%m-%d"),
                "Weekday": day_date.strftime("%A"),
                "Target Hours": daily_hours,
                "Topic Day": f"{d + 1}/{duration}",
                "Topic": topic,
                "Status": "Pending"
            })
            day_counter += 1

    # 1. Topic study periods (how long each topic takes to finish)
    print("=" * 85)
    print("📅 TOPIC STUDY PERIODS (Duration & Timeline to Finish Each Topic):")
    print("=" * 85)
    for p in topic_periods:
        print(f"• Topic {p['Topic #']}: {p['Topic']:<24} | Period: {p['Start Date']} to {p['End Date']} ({p['Duration']}, {p['Total Hours']})")
    print("=" * 85 + "\n")

    # 2. Print the WHOLE day-by-day table without any truncation
    print("📋 FULL DAY-BY-DAY STUDY SCHEDULE (All Days):")
    print(f"{'Day':<5} {'Date':<12} {'Weekday':<11} {'Hours':<8} {'Topic Day':<11} {'Topic'}")
    print("-" * 80)
    for row in schedule_rows:
        print(f"{row['Day']:<5} {row['Date']:<12} {row['Weekday']:<11} {row['Target Hours']:<8} {row['Topic Day']:<11} {row['Topic']}")

    # Store full schedule in a DataFrame for subsequent steps
    df_schedule = pd.DataFrame(schedule_rows)
else:
    print("Please ensure topics and exam date are properly set.")
    df_schedule = pd.DataFrame(columns=["Day", "Date", "Weekday", "Target Hours", "Topic Day", "Topic", "Status"])

📅 TOPIC STUDY PERIODS (Duration & Timeline to Finish Each Topic):
• Topic 1: Introduction & Syntax    | Period: 2026-10-16 to 2026-10-16 (1 days, 135.0 hrs)
• Topic 2: Data Structures          | Period: 2026-10-17 to 2026-10-17 (0 days, 0.0 hrs)
• Topic 3: OOP                      | Period: 2026-10-17 to 2026-10-17 (0 days, 0.0 hrs)
• Topic 4: File Handling            | Period: 2026-10-17 to 2026-10-17 (0 days, 0.0 hrs)
• Topic 5: Project & Revision       | Period: 2026-10-17 to 2026-10-17 (0 days, 0.0 hrs)

📋 FULL DAY-BY-DAY STUDY SCHEDULE (All Days):
Day   Date         Weekday     Hours    Topic Day   Topic
--------------------------------------------------------------------------------
1     2026-10-16   Friday      135.0    1/1         Introduction & Syntax


### Step 6: Log Daily Progress

In [61]:
# Clean up typo column if previously created in earlier runs
if "Satus" in df_schedule.columns:
    df_schedule = df_schedule.drop(columns=["Satus"])

# Example situation: User misses Day 1 and completes Day 2
if len(df_schedule) >= 2:
    df_schedule.loc[0, "Status"] = "Missed"
    df_schedule.loc[1, "Status"] = "Completed"
elif len(df_schedule) == 1:
    df_schedule.loc[0, "Status"] = "Missed"

print("Updated Schedule Status: ")
display(df_schedule.head(5))

Updated Schedule Status: 


,Day,Date,Weekday,Target Hours,Topic Day,Topic,Status
0,1,2026-10-16,Friday,135.0,1/1,Introduction & Syntax,Missed


### Step 7: Adaptive Rescheduling

In [ ]:
def reschedule(df, exam_date, max_daily_cap=4.0):
    """
    Recalculates target daily hours and redistributes missed/pending topics
    across remaining days before the exam.
    """
    today_date = datetime.today().date()

    # Identify available future/today days that are still Pending
    pending_days_mask = (pd.to_datetime(df["Date"]).dt.date >= today_date) & (df["Status"] == "Pending")
    available_pending_days = df[pending_days_mask]
    num_available_days = len(available_pending_days)

    # All uncompleted work (both missed tasks and pending tasks)
    pending_or_missed = df[df["Status"].isin(["Pending", "Missed"])]
    uncompleted_hours = pending_or_missed["Target Hours"].sum()

    if num_available_days == 0:
        print("⚠️ No available study days left before the exam! NOW GET SERIOUS RN")
        return df

    # Redistribute uncompleted study hours across remaining available study days
    newDailyHours = round(uncompleted_hours / num_available_days, 2)

    print("=" * 45)
    print("🔄 REPLANNING SUMMARY")
    print("Available Study Days Left:", num_available_days)
    print(f"Lost Study Hours: {uncompleted_hours:.1f} hrs")
    print(f"New Adjusted Daily Target: {newDailyHours:.2f} hrs/day")

    # Feasibility warning check
    if newDailyHours > max_daily_cap:
        print(f"⚠️ FEASIBILITY ALERT: Required {newDailyHours} hrs/day exceeds your limit ({max_daily_cap} hrs/day)!")
    print("=" * 45)

    # Update future pending days with the new rate
    df.loc[pending_days_mask, "Target Hours"] = newDailyHours
    return df

df_schedule = reschedule(df_schedule, exdate)
print("\nUpdated Schedule After Rescheduling:")
display(df_schedule.head(5))

⚠️ No available study days left before the exam! NOW GET SERIOUS RN

Updated Schedule After Rescheduling:


,Day,Date,Weekday,Target Hours,Topic Day,Topic,Status
0,1,2026-10-16,Friday,135.0,1/1,Introduction & Syntax,Missed
